# Prodigy InfoTech Task-03 — Dogs vs Cats using SVM

This notebook demonstrates classical image classification using HOG features and an RBF-kernel Support Vector Machine.

The dataset is the Kaggle Dogs vs Cats competition dataset. Run `python download_data.py` first if the data is not available locally.

In [ ]:
from pathlib import Path
import re
import cv2
import joblib
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

TRAIN_DIR = Path('../data/raw/train')
IMAGE_SIZE = (64, 64)
MAX_PER_CLASS = 2500
RANDOM_STATE = 42

## 1. Load image paths and labels

In [ ]:
cat_files = sorted(TRAIN_DIR.glob('cat.*.jpg'))[:MAX_PER_CLASS]
dog_files = sorted(TRAIN_DIR.glob('dog.*.jpg'))[:MAX_PER_CLASS]
files = cat_files + dog_files
labels = np.array([0]*len(cat_files) + [1]*len(dog_files))
print('Cats:', len(cat_files), 'Dogs:', len(dog_files), 'Total:', len(files))

## 2. Extract HOG features

In [ ]:
def extract_hog(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, IMAGE_SIZE, interpolation=cv2.INTER_AREA)
    return hog(img, orientations=9, pixels_per_cell=(8,8), cells_per_block=(2,2), block_norm='L2-Hys')

X = np.array([extract_hog(p) for p in files], dtype=np.float32)
print('Feature shape:', X.shape)

## 3. Train/test split and SVM training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.20, random_state=RANDOM_STATE, stratify=labels
)

model = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=RANDOM_STATE))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['Cat', 'Dog']))

## 4. Confusion matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['Cat','Dog'])
plt.title('Dogs vs Cats — SVM Confusion Matrix')
plt.show()

## 5. Save model

In [ ]:
Path('../models').mkdir(exist_ok=True)
joblib.dump(model, '../models/svm_pipeline.joblib')
print('Saved: ../models/svm_pipeline.joblib')

## Conclusion

The model uses HOG to represent image edges and shapes and an RBF SVM for binary classification. The final accuracy depends on the number of training images and the local environment. The saved model can be used by `predict.py` or the Flask web application.